In [56]:
from xmlrpc.client import boolean

import duckdb
import pandas as pd
import numpy as np

In [57]:
DATABASE_PATH = "../healthcare.duckdb"

connection = duckdb.connect(DATABASE_PATH)

df_raw = connection.execute("SELECT * FROM raw_analytical_dataset").fetch_df()

connection.close()

In [58]:
df_clean = df_raw.copy()

In [59]:
# Gender Category
gender_mapping = {
    "male" : "Male",
    "MALE" : "Male",
    "M" : "Male",

    "female" : "Female",
    "F" : "Female"
}
df_clean["gender"] = (df_clean["gender"].replace(gender_mapping))

In [60]:
df_clean["gender"].unique()

<StringArray>
['Male', 'Female', nan]
Length: 3, dtype: str

In [61]:
# Blood Group Category
blood_mapping = {
    "O positive" : "O+",
    "B Plus" : "B+",
    "AB Plus" : "AB+",
    "a+" : "A+",
    "A Plus" : "A+",
    "A positive" : "A+"
}
df_clean["blood_group"] = df_clean['blood_group'].replace(blood_mapping)

In [62]:
df_clean['blood_group'].unique()

<StringArray>
['AB+', 'AB-', 'O+', 'A-', 'B-', nan, 'O-', 'A+', 'B+']
Length: 9, dtype: str

In [63]:
# Invalid ages (age < 0 and age > 120) - Convert to missing values
df_clean['age'] = df_clean['age'].mask(~df_clean['age'].between(0,120))


In [64]:
invalid_age_count = df_clean[(df_clean['age'] < 0) | (df_clean['age'] > 120)]
print(len(invalid_age_count))

0


In [65]:
# Invalid height ( height < 50cm and > 250cm) - Mark as missing
df_clean['height_cm'] = df_clean['height_cm'].mask(~df_clean['height_cm'].between(50,250))

In [66]:
invalid_height_count = df_clean[(df_clean['height_cm'] < 50) | (df_clean['height_cm'] > 250)]
print(len(invalid_height_count))

0


In [67]:
# Invalid weight (doing the same - <3 >300)
df_clean['weight_kg'] = df_clean['weight_kg'].mask(~df_clean['weight_kg'].between(3,300))

In [68]:
invalid_weight_count = df_clean[(df_clean['weight_kg'] < 3) | (df_clean['weight_kg'] > 300)]
print(len(invalid_weight_count))

0


In [69]:
# Recalculating BMI
df_clean['bmi'] = (df_clean['weight_kg'] / (df_clean['height_cm'] / 100)**2)

In [70]:
df_clean['bmi'] = df_clean['bmi'].round(2)

In [71]:
# -ve LOS values
neg_LOS_value = df_clean[df_clean['length_of_stay'] < 0]
print(len(neg_LOS_value))

526


In [72]:
print(df_clean[df_clean['length_of_stay'] < 0]['length_of_stay'].value_counts())

length_of_stay
-10    274
-5     252
Name: count, dtype: int64


In [73]:
# This shows that -5, -10 can be actual LOS for patients and can be a typo... But checking it with room_cost sets things clear as if Room cost is > 0 then the patient has stayed and we use abs() otherwise we do NaN.

df_clean.loc[df_clean['length_of_stay'] < 0,["patient_name", "length_of_stay","room_cost","treatment_type", "severity"]].head(10)

,patient_name,length_of_stay,room_cost,treatment_type,severity
500,Krisha Dyal,-5,67084.58,Therapy,Moderate
961,Urmi Dhar,-5,89821.54,Therapy,Mild
1427,Isha Lad,-5,87656.77,Medication,Moderate
1882,Onkar Sur,-5,28933.95,Surgery,NaN
4012,Udarsh Kade,-10,95082.67,Medication,Mild
4568,Lila Bir,-10,97284.81,Medication,Severe
4943,Devika Chada,-5,9830.53,Surgery,Mild
5949,Edhitha Raman,-5,83976.46,Medication,Moderate
6067,Arunima Borde,-5,7633.32,Observation,Severe
6095,Radhika Jhaveri,-10,85461.06,Medication,Moderate


In [74]:
all_pos_room_cost = (df_clean['room_cost'] > 0).all()
print(all_pos_room_cost) # all room_cost are +ve

True


In [75]:
df_clean['length_of_stay'] = df_clean['length_of_stay'].abs()

In [76]:
# Invalid sleep values (<0 and >24)
df_clean['sleep_hours'] = df_clean['sleep_hours'].mask(~df_clean['sleep_hours'].between(0,24))

In [77]:
invalid_sleep_hours = df_clean[(df_clean['sleep_hours'] < 0) | (df_clean['sleep_hours'] > 24)]
print(len(invalid_sleep_hours))

0


In [78]:
# Satisfaction score out of range (1-5) - NaN
df_clean['satisfaction_rating'] = df_clean['satisfaction_rating'].mask(~df_clean['satisfaction_rating'].between(1,5))

In [79]:
invalid_satisfaction_score = df_clean[(df_clean['satisfaction_rating'] < 1) | (df_clean['satisfaction_rating'] > 5)]
print(len(invalid_satisfaction_score))

0


In [80]:
# Same goes for Recommendation score (1-10) - NaN
df_clean['recommendation_score'] = df_clean['recommendation_score'].mask(~df_clean['recommendation_score'].between(1,10))

In [81]:
invalid_recommendations = df_clean[(df_clean['recommendation_score'] < 1) | (df_clean['recommendation_score'] > 10)]
print(len(invalid_recommendations))

0


In [82]:
# Recalculating -ve bill amount by adding the bill parts.
df_clean['bill_amount'] = (df_clean['medicine_cost']
                           + df_clean['lab_cost'] +
                           df_clean['room_cost'] +
                           df_clean['doctor_fee'] +
                           df_clean['other_charges']
)

In [83]:
neg_bill_count = df_clean[df_clean['bill_amount'] < 0]
print(len(neg_bill_count)) # Fixed

0


In [85]:
# Patient paid can also not be -ve
df_clean.loc[df_clean["patient_paid"] < 0,"patient_paid"] = np.nan

In [86]:
neg_bill_paid_count = df_clean[df_clean['patient_paid'] < 0]
print(len(neg_bill_paid_count))

0


In [87]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 51 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   patient_id            500000 non-null  str    
 1   patient_name          500000 non-null  str    
 2   age                   499037 non-null  float64
 3   gender                475124 non-null  str    
 4   blood_group           475120 non-null  str    
 5   height_cm             474288 non-null  float64
 6   weight_kg             474532 non-null  float64
 7   bmi                   450108 non-null  float64
 8   patient_city          500000 non-null  str    
 9   patient_state         500000 non-null  str    
 10  pincode               500000 non-null  int64  
 11  marital_status        475000 non-null  str    
 12  primary_diagnosis     500000 non-null  str    
 13  severity              475000 non-null  str    
 14  admission_type        500000 non-null  str    
 15  treatment_t

In [88]:
df_clean.describe().T

,count,mean,std,min,25%,50%,75%,max
age,499037.0,50.463244,28.889445,1.00,25.0000,50.000,76.0000,120.000
height_cm,474288.0,159.986019,23.187666,120.00,139.9000,159.900,180.0000,250.000
weight_kg,474532.0,87.684191,36.578371,25.00,56.3000,87.500,118.9000,300.000
bmi,450108.0,36.538381,19.275771,4.06,21.5300,33.510,47.6300,206.270
pincode,500000.0,550342.255060,259520.609031,100000.00,325748.0000,550544.500,774971.0000,999992.000
length_of_stay,500000.0,5.042556,6.696868,1.00,2.0000,3.000,6.0000,200.000
experience_years,500000.0,18.362250,10.130866,1.00,9.0000,19.000,27.0000,35.000
bed_capacity,500000.0,496.539296,274.849790,52.00,288.0000,483.000,732.0000,998.000
medicine_cost,500000.0,25244.352162,14283.775187,500.06,12888.5275,25260.650,37610.5375,49999.610
lab_cost,500000.0,15151.033564,8573.177024,300.01,7731.9950,15149.275,22577.9200,29999.930


In [89]:
# Finding missing values, their count and percentage
missing_after_cleaning = pd.DataFrame({
    "missing_count" : df_clean.isna().sum(),
    "missing_percentage" : df_clean.isna().mean()*100
})

missing_after_cleaning = (missing_after_cleaning.sort_values("missing_percentage", ascending=False))

missing_after_cleaning[missing_after_cleaning['missing_count'] > 0]

,missing_count,missing_percentage
bmi,49892,9.9784
satisfaction_rating,27382,5.4764
sleep_hours,26057,5.2114
height_cm,25712,5.1424
weight_kg,25468,5.0936
severity,25000,5.0000
marital_status,25000,5.0000
diet_type,25000,5.0000
smoker,25000,5.0000
insurance_provider,25000,5.0000


In [90]:
# Age (963 missing) - Age is a fundamental demographic variable, and we don't have reliable variable to reconstruct someone's age - So we keep it as NaN only.

In [91]:
# Gender (24,876 missing) - Keep it "unknown"
df_clean['gender'] = df_clean['gender'].fillna("Unknown")

In [92]:
# Same goes for blood group, we cannot infer a person's blood group (24,880)
df_clean['blood_group'] = df_clean['blood_group'].fillna("Unknown")

In [93]:
# Same for marital status (25,000)
df_clean['marital_status'] = df_clean['marital_status'].fillna("Unknown")

In [94]:
# Same for severity
df_clean['severity'] = df_clean['severity'].fillna("Unknown")

In [95]:
# Same for these columns as well
df_clean['alcohol_consumption'] = df_clean['alcohol_consumption'].fillna("Unknown")

df_clean['exercise_frequency'] = df_clean['exercise_frequency'].fillna("Unknown")

df_clean['diet_type'] = df_clean['diet_type'].fillna("Unknown")

In [96]:
# Since smoker is a Yes/No we will keep it as boolean only.
df_clean['smoker'] = df_clean['smoker'].astype("boolean")

In [97]:
# For insurance provider, there were 2 values None and NaN .... Both are different
df_clean['insurance_provider'] = df_clean['insurance_provider'].fillna("Unknown")

In [98]:
# Sleep hours we cna put as median

In [99]:
df_clean.loc[df_clean['patient_paid'].isna(),['patient_id','bill_amount','insurance_provider','insurance_coverage','patient_paid','payment_status','payment_method']].head(10)

,patient_id,bill_amount,insurance_provider,insurance_coverage,patient_paid,payment_status,payment_method
78,PAT493865,101946.77,Care Health,74936.69,NaN,Partially Paid,Debit Card
260,PAT499669,85810.23,ICICI Lombard,48423.81,NaN,Pending,Net Banking
725,PAT132264,143782.96,Star Health,128300.12,NaN,Paid,UPI
803,PAT135041,94045.26,Care Health,48507.16,NaN,Pending,Net Banking
1151,PAT146277,142464.46,None,0.00,NaN,Paid,Credit Card
1204,PAT147936,63739.77,None,0.00,NaN,Partially Paid,Cash
1844,PAT169509,83309.02,Star Health,60886.08,NaN,Paid,Cash
2153,PAT179155,130353.29,HDFC Ergo,69846.25,NaN,Paid,Debit Card
2217,PAT180779,159537.65,None,0.00,NaN,Partially Paid,Cash
2270,PAT182317,125053.87,Unknown,66895.38,NaN,Paid,UPI


In [100]:
df_clean.groupby('payment_status',dropna=False)[['bill_amount','insurance_coverage','patient_paid']].agg(['count','mean','min'])

bill_amount                          insurance_coverage  \
                     count           mean       min              count   
payment_status                                                           
Paid                349654  111116.031768   7459.47             349654   
Partially Paid      100039  111146.843209  12226.10             100039   
Pending              50307  110891.383231  13574.99              50307   

                                  patient_paid                         
                        mean  min        count          mean      min  
payment_status                                                         
Paid            54913.228415  0.0       347890  56592.339873  1287.15  
Partially Paid  54977.305514  0.0        99550  56557.709236  2243.95  
Pending         54609.637857  0.0        50060  56657.534336  2083.03

In [101]:
# At this process, data is clean enough to start performing EDA as many NaN values cannot be filled without finding proper relationships between columns (After EDA, in feature engineering we'll fill the values)

In [102]:
df_clean.to_csv("../data/cleaned_healthcare_data.csv",index=False)